In [ ]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# #Importing Model Data
    
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
# true_time=data['time']
# # parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
# times=data['time'].values/(1e9 * 60); times=times.astype(float);
# Np_str='125e3'
# #Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,140+1))
# # parcel=parcel.isel(time=np.arange(0,140+1))
# res='1km'

dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
true_time=data['time']
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
Np_str='1e6'
#Restricts the timesteps of the data from timesteps0 to 140
res='1km'
job_array=False;index_adjust=0
ocean_fraction=0.25

# dx = 1km; Np = 50M
#Importing Model Data
check=False
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'

# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# # # parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# # # parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [ ]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

times=data['time'].values/(1e9 * 60); times=times.astype(float);
minutes=1/times[1] #1 / minutes per timestep = timesteps per minute
kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1)

In [ ]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [ ]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(parcel['xh']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=2
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

In [ ]:
#Indexing Array with JobArray
parcel=parcel.isel(xh=slice(start_job,end_job))
#(for 150_000_000 parcels use 500-1000 jobs)

In [ ]:
# Reading Back Data Later
##############
def make_data_dict(var_names,read_type):
    if read_type=='h5py':
        with h5py.File(in_file, 'r') as f:
            data_dict = {var_name: f[var_name][:,start_job:end_job] for var_name in var_names}
            
    elif read_type=='xarray':
        in_data = xr.open_dataset(
            in_file,
            engine='h5netcdf',
            phony_dims='sort',
            chunks={'phony_dim_0': 100, 'phony_dim_1': 1_000_000} 
        )
        data_dict = {k: in_data[k][:,start_job:end_job].compute().data for k in var_names}
    return data_dict

# read_type='xarray'
read_type='h5py'

In [ ]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min.h5'
# in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_1min.h5'
# in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_1min_50M.h5'

var_names = ['A_g', 'A_c', 'W', 'QCQI', 'Z', 'Y', 'X', 'z']
data_dict = make_data_dict(var_names,read_type)
A_g, A_c, W, QCQI, Z, Y, X, parcel_z = (data_dict[k] for k in var_names)

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory()

In [ ]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'VARS_binary_array_{res}_{Np_str}_5min.h5'

var_names = ['VMF_c','VMF_g']
data_dict = make_data_dict(var_names,read_type)
VMF_c, VMF_g = [data_dict[k] for k in var_names]
check_memory()

In [ ]:
################################################################################

In [ ]:
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'

# Reading Back Data Later
##################################################################
#DOMAIN SUBSETTING
def DOMAIN_SUBSET(out_arr):
    print(f'length before: {len(out_arr)}')

    #SETTING UP
    ###########
    ocean_percent=2/8
    left_to_coast=data['xh'][0]+(data['xh'][-1]-data['xh'][0])*ocean_percent
    
    where_coast_xh=np.where(data['xh']>=left_to_coast)[0][0]#-25
    where_coast_xf=np.where(data['xf']>=left_to_coast)[0][0]#-25
    end_xh=len(data['xh'])-1-50
    end_xf=len(data['xf'])-1-50
    
    print(f'x in {0}:{where_coast_xh-1} FOR SEA')
    print(f'x in {where_coast_xh}:{end_xh} FOR LAND')
    
    t_start=36 
    t_end=len(data['time']) # if res=='250m': t_end=410
    print(f't in {t_start}:end (8 hours)')

    #SUBSETTING CODE
    ################
    t,p=out_arr[:,1],out_arr[:,0]
    if 'job_array' in globals():
        p -= index_adjust

    #GETTING X VALUES OF EACH PARCEL 
    fancy_index=True
    # fancy_index=False
    if fancy_index==True:
        xs=X[t,p] #FANCY INDEXING
    elif fancy_index==False: #(slightly longer, but avoids loading X into memory)
        with h5py.File(in_file, 'r') as f:
            xs=[]
            for i, j in zip(t, p):
                xs.append(f['X'][i, j])
            xs=np.array(xs)
    ################
    
    out_arr=out_arr[np.where((xs>=where_coast_xh)&(xs<=end_xh))]
    out_arr=out_arr[np.where(out_arr[:,1]<=t_end)]

    print(f'==> length after: {len(out_arr)}'+'\n')
    return out_arr

In [ ]:
#LOADING BACK IN
def load_file():
    in_file=dir+f'Project_Algorithms/Tracking_Algorithms/parcel_tracking_{res}_{t_res}_{Np_str}.h5'
    with h5py.File(in_file, 'r') as hf:
        out_arr=hf['out_arr'][:]
        save_arr=hf['save_arr'][:]
        save2_arr=hf['save2_arr'][:]
    return out_arr,save_arr,save2_arr
[out_arr,save_arr,save2_arr]=load_file()

print('list of first 10 ignored parcels');
print(f'there are a total of {len(out_arr)} CL parcels and {len(save_arr)} nonCL parcels'+'\n')

if 'job_array' in globals():
    #APPLYING JOB_ARRAY TO PARCEL NUMBER
    ####################################
    def job_filter(arr):
        return arr[(arr[:,0]>=start_job)&(arr[:,0]<end_job)]
    print('Applying Job Array'+'\n')
    out_arr=job_filter(out_arr)
    save_arr=job_filter(save_arr)

#CHOOSING UNIQUE INDEXES
###############################################################################
def remove_duplicates(arr):
    lst = []
    unique_values, counts = np.unique(arr[:, 0], return_counts=True)
    duplicates = unique_values[counts > 1]
    for elem in duplicates:
        idx = np.where(arr[:, 0] == elem)[0]
        extras = idx[np.where(arr[idx, 1] != np.min(arr[idx, 1]))]
        lst.extend(extras)
    mask = np.ones(len(arr), dtype=bool)
    mask[lst] = False
    return arr[mask]
out_arr=remove_duplicates(out_arr)
save_arr=remove_duplicates(save_arr)
###############################################################################

############################################################
#SUBSETTING
subset=True
if subset==True:
    out_arr=DOMAIN_SUBSET(out_arr)
    save_arr=DOMAIN_SUBSET(save_arr)
############################################################

ALL_out_arr=out_arr.copy(); ALL_save_arr=save_arr.copy()

In [ ]:
#SHALLOW

def ddt(f,dt=1):
    ddx = (
            f[1:  ]
            -
            f[0:-1]
        ) / (
        2 * dt
    )
    return ddx

#search for deep convective parcels within lagrangian tracking output     
##############################################################
def SHALLOW_threshold(zthresh,type):

    if type=='CL':
        out_arr=ALL_out_arr.copy()
    elif type=='nonCL':
        out_arr=ALL_save_arr.copy()
    
    deep_out_ind=[]; extendrange=[]
    times=data['time'].values/(1e9 * 60); times=times.astype(float);
    for ind in range(len(out_arr)): 
        if np.mod(ind,5000)==0: print(f'{ind}/{len(out_arr)}')
        #CHECK TO SSEE IF NEXT MOST LOCAL TIME MAX GOES ABOVE ZTHRESHS 

        #Get Ascending Range Past LFC For Maximum 120 Minutes Simulation Time
        nummins=120; numsteps=int(nummins/times[1])
        aboverange=np.arange(out_arr[ind,2],out_arr[ind,2]+numsteps,1) #range of times between current time and numsteps later
        aboverange=aboverange[aboverange<len(data['time'])] #caps out at max time
        above=parcel_z[aboverange,out_arr[ind,0]-index_adjust]/1000 #JOBARRAY INDEX_ADJUST
    
        #Takes The time derivative 
        ddx=ddt(above)

        #Checks whether the Local Time Max Is Located Above zthresh
        signs = np.sign(ddx)
        signs_diff=np.diff(signs)
        local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here
        if len(local_maxes)==0:
            local_maxes=[0]
        elif np.any(above[local_maxes[0]]<=zthresh): #< for SHALLOW, > for DEEP
            extendrange.append(local_maxes[0]) #save to extend xlim of plot later
            deep_out_ind.append(ind)

    #SUBSET OUT FOR FINAL RESULT
    out_arr=out_arr[deep_out_ind,:]
    # print(f'> {zthresh} km. {len(out_arr)} leftover parcels')
    return out_arr#, extendrange
    # print(out_arr)
##############################################################

convectivelevel=4 #4km
SHALLOW_out_arr=SHALLOW_threshold(convectivelevel,type='CL')
SHALLOW_save_arr=SHALLOW_threshold(convectivelevel,type='nonCL')

print('list of first 10 SBZ parcels'); print(out_arr[:15])
print(f'there are a total of {len(SHALLOW_out_arr)} CL parcels and {len(SHALLOW_save_arr)} nonCL parcels')

In [ ]:
#DEEP

#search for deep convective parcels within lagrangian tracking output     
##############################################################
def DEEP_threshold(zthresh,type):
    if type=='CL':
        out_arr=ALL_out_arr.copy()
    elif type=='nonCL':
        out_arr=ALL_save_arr.copy()
    
    deep_out_ind=[]; extendrange=[]
    times=data['time'].values/(1e9 * 60); times=times.astype(float);
    for ind in range(len(out_arr)): 
        if np.mod(ind,5000)==0: print(f'{ind}/{len(out_arr)}')
        #CHECK TO SSEE IF NEXT MOST LOCAL TIME MAX GOES ABOVE ZTHRESHS 

        #Get Ascending Range Past LFC For Maximum 120 Minutes Simulation Time
        nummins=120; numsteps=int(nummins/times[1])
        aboverange=np.arange(out_arr[ind,2],out_arr[ind,2]+numsteps,1) #range of times between current time and numsteps later
        aboverange=aboverange[aboverange<len(data['time'])] #caps out at max time
        above=parcel_z[aboverange,out_arr[ind,0]-index_adjust]/1000 #JOBARRAY INDEX_ADJUST
        
        #Takes The time derivative 
        ddx=ddt(above)

        #Checks whether the Local Time Max Is Located Above zthresh
        signs = np.sign(ddx)
        signs_diff=np.diff(signs)
        local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here
        if len(local_maxes)==0:
            local_maxes=[0]
        
        if np.any(above[local_maxes[0]]>=zthresh): #< for SHALLOW, > for DEEP
            extendrange.append(local_maxes[0]) #save to extend xlim of plot later
            deep_out_ind.append(ind)

    #SUBSET OUT FOR FINAL RESULT
    out_arr=out_arr[deep_out_ind,:]
    # print(f'> {zthresh} km. {len(out_arr)} leftover parcels')
    return out_arr#, extendrange
    # print(out_arr)
##############################################################

convectivelevel=6 #4km
DEEP_out_arr=DEEP_threshold(convectivelevel,type='CL')
DEEP_save_arr=DEEP_threshold(convectivelevel,type='nonCL')

print('list of first 10 SBZ parcels'); print(out_arr[:15])
print(f'there are a total of {len(DEEP_out_arr)} CL parcels and {len(DEEP_save_arr)} nonCL parcels')

In [ ]:
#FIND WHERE PARCEL LEAVES UPDRAFT (FOR ALL AND DEEP PARCELS)
def find_after_time(out_arr):
    wthresh=0.5;qcqithresh=1e-6
    
    after_array=np.zeros(len(out_arr),dtype=int)
    for count,out_row in enumerate(out_arr): 
        
        p=out_row[0]
        t2=out_row[2]
        
        wthresh=0.5;qcqithresh=1e-6
        after=np.where((W[t2:,p-index_adjust]<wthresh)|(QCQI[t2:,p-index_adjust]<qcqithresh)) #JOBARRAY INDEX_ADJUST
        if len(after[0])!=0:
            after_array[count]=after[0][0]
    return after_array

ALL_out_after_array=find_after_time(ALL_out_arr)
SHALLOW_out_after_array=np.zeros(len(SHALLOW_out_arr),dtype=int)
DEEP_out_after_array=find_after_time(DEEP_out_arr)

ALL_save_after_array=find_after_time(ALL_save_arr)
SHALLOW_save_after_array=np.zeros(len(SHALLOW_save_arr),dtype=int)
DEEP_save_after_array=find_after_time(DEEP_save_arr)

# # Create a figure and specify gridspec layout
# fig = plt.figure(figsize=(8, 4))
# gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1])  # 1 row, 2 columns
# ax0 = plt.subplot(gs[0])
# ax0.hist(ALL_after_array, bins=25)
# ax0.set_xlabel('time')
# ax0.set_ylabel('count')
# ax0.set_title('ALL_after')
# ax1 = plt.subplot(gs[1])
# ax1.hist(DEEP_after_array, bins=25)
# ax1.set_xlabel('time')
# ax1.set_ylabel('count')
# ax1.set_title('DEEP_after')
# plt.tight_layout()

# # Multiply x-ticks by 5
# xticks = ax0.get_xticks()  # Get current x-tick positions
# ax0.set_xticks(xticks)  # Set the same ticks
# ax0.set_xticklabels([int(x * 5) for x in xticks])  # Multiply tick labels by 5

# # Multiply x-ticks by 5
# xticks = ax1.get_xticks()  # Get current x-tick positions
# ax1.set_xticks(xticks)  # Set the same ticks
# ax1.set_xticklabels([int(x * 5) for x in xticks])  # Multiply tick labels by 5

# ax0.set_xlim(left=0);ax1.set_xlim(left=0)

In [ ]:
#FINDING MEAN CLOUD BASE 
zh=data['zh'].values
w_thresh2=0.5
qcqi_thresh=1e-6
type='all'


if type=='all':
    out_arr=ALL_out_arr.copy()
elif type=='deep':
    out_arr=DEEP_out_arr.copy()
elif type=='shallow':
    out_arr=SHALLOW_out_arr.copy()

zhs=data['zh'].values
profile_array =np.zeros((len(zhs), 2)) #column 1: var, column 2: counter, column 3: list of zhs
profile_array[:,1]=zhs;

# cloudbase_lst=[]
after=4 #20 minutes
for row in range(out_arr.shape[0]):
    if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
    p=out_arr[row,0]
    
    # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
    ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
    ts = np.arange(out_arr[row, 1], ts_end)
    
    zs=Z[ts,p-index_adjust] #JOBARRAY INDEX_ADJUST
    ys=Y[ts,p-index_adjust] #JOBARRAY INDEX_ADJUST
    xs=X[ts,p-index_adjust] #JOBARRAY INDEX_ADJUST

    ws=W[ts,p-index_adjust] #JOBARRAY INDEX_ADJUST
    qcqis=QCQI[ts,p-index_adjust] #JOBARRAY INDEX_ADJUST
    where=np.where((ws>=w_thresh2) & (qcqis>=qcqi_thresh))
    profile_array[zs[where],0]+=1
del after
# print(np.mean(cloudbase_lst))
# print(np.min(cloudbase_lst))
# plt.hist(cloudbase_lst,bins=40,orientation='horizontal');
all_cloudbase=zh[np.where(profile_array[:,0]!=0)[0][0]]
print(all_cloudbase)

In [ ]:
#ALL/DEEP/SHALLOW CL vs non-CL Tracked Parcel Plots
################################################################################

In [ ]:
def averaged_profiles(profile): 
    out_var=profile[ (profile[:, 1] != 0)]; #gets rid of rows that have no data
    # out_var=np.array([out_var[:, 0] / out_var[:, 1], out_var[:, 2]]).T #divides the data column by the counter column
    out_var=np.array([out_var[:, 0], out_var[:, 2]]).T 
    return out_var

In [ ]:
def CL_tracked_profile(var_data,type):
    if type=='all':
        out_arr=ALL_out_arr.copy()
        after_array=ALL_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_out_arr.copy()
        after_array=SHALLOW_out_after_array
    elif type=='deep':
        out_arr=DEEP_out_arr.copy()
        after_array=DEEP_out_after_array

    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=after_array[row]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        
        vars=var_data[ts,p-index_adjust]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array

print('ALL')
ALL_profile_array_VMF_c=CL_tracked_profile(VMF_c,type='all')
ALL_profile_array_VMF_g=CL_tracked_profile(VMF_g,type='all')
print('SHALLOW')
SHALLOW_profile_array_VMF_c=CL_tracked_profile(VMF_c,type='shallow')
SHALLOW_profile_array_VMF_g=CL_tracked_profile(VMF_g,type='shallow')
print('DEEP')
DEEP_profile_array_VMF_c=CL_tracked_profile(VMF_c,type='deep')
DEEP_profile_array_VMF_g=CL_tracked_profile(VMF_g,type='deep')

In [ ]:
#MAKING PROFILE ARRAY

def nonCL_tracked_profile(var_data,type):
    if type=='all':
        out_arr=ALL_save_arr.copy()
        after_array=ALL_save_after_array
    elif type=='shallow':
        out_arr=SHALLOW_save_arr.copy()
        after_array=SHALLOW_save_after_array
    elif type=='deep':
        out_arr=DEEP_save_arr.copy()
        after_array=DEEP_save_after_array
    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        after=after_array[row]
        p=out_arr[row,0]

        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        
        vars=var_data[ts,p-index_adjust]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array

print('ALL')
nonCL_ALL_profile_array_VMF_c=nonCL_tracked_profile(VMF_c,type='all')
nonCL_ALL_profile_array_VMF_g=nonCL_tracked_profile(VMF_g,type='all')
print('SHALLOW')
nonCL_SHALLOW_profile_array_VMF_c=nonCL_tracked_profile(VMF_c,type='shallow')
nonCL_SHALLOW_profile_array_VMF_g=nonCL_tracked_profile(VMF_g,type='shallow')
print('DEEP')
nonCL_DEEP_profile_array_VMF_c=nonCL_tracked_profile(VMF_c,type='deep')
nonCL_DEEP_profile_array_VMF_g=nonCL_tracked_profile(VMF_g,type='deep')

In [ ]:
#SAVING

# Define categories and variables
types = ["ALL", "SHALLOW", "DEEP"]
variables = ["VMF_c","VMF_g"]

# Dictionary to store all variables
tracked_profiles = {}

# Populate all profiles in one loop
for type in types:
    for var in variables:
        tracked_profiles[f"{type}_profile_array_{var}"] = eval(f"{type}_profile_array_{var}")
        tracked_profiles[f"nonCL_{type}_profile_array_{var}"] = eval(f"nonCL_{type}_profile_array_{var}")

# Save all variables in an HDF5 file
dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out2/'
output_file=dir2+f"CL_nonCL_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min_{job_id}.h5"
with h5py.File(output_file, "w") as h5f:
    for name, profile_data in tracked_profiles.items():
        h5f.create_dataset(name, data=profile_data)
print('done')


In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [ ]:
if recombine==True:
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["VMF_c","VMF_g"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_profile_array_{var}")

In [ ]:
if recombine==True:
    dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out2/'
    
    
    #MAKING OUTPUT FILE PATH
    output_file=dir2+f"CL_nonCL_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min.h5"
    
    #MAKING PROFILES DICTIONARY
    zhs = data['zh'].values
    profiles = {}  # Store profiles for all variables
    for var in vars_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs 
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
    
        #CALLING IN DATA
        input_file=dir2+f"CL_nonCL_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min_{job_id}.h5"
    
        #COMPILING PROFILES
        with h5py.File(input_file, 'r') as f:
            for var in vars_list:
                profiles[var][:,0:1+1]+=f[f'{var}'][:,0:1+1]
    
    #SAVING INTO FINAL FORM
    with h5py.File(output_file, 'w') as f:
        for var in profiles:
            profile_var = profiles[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

In [ ]:
##############################
#PLOTTING
plotting=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# plotting=True

In [ ]:
if plotting==True:
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["VMF_c","VMF_g"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_profile_array_{var}")
            vars_list.append(f"nonCL_{t}_profile_array_{var}")
    
    # Define directory and output file path
    dir2 = dir + 'Project_Algorithms/Tracked_Profiles/job_out2/'
    output_file = dir2+f"CL_nonCL_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min.h5"
    
    # Open the HDF5 file and read the stored datasets into dynamically named variables
    with h5py.File(output_file, 'r') as f:
        for var in vars_list:
            globals()[var] = f[f'{var}'][:]
            globals()[var][:,0]/=Nt #*#

In [ ]:
def averaged_profiles(profile):
    out_var = profile[(profile[:, 1] != 0)]  # gets rid of rows that have no data
    # out_var = np.array([out_var[:, 0] / out_var[:, 1], out_var[:, 2]]).T  # divides the data column by the counter column
    
    Nt=len(data['time']);Ny=len(data['yh']);Nx=len(data['xh'])
    out_var[:,0]/=(Ny*Nx*Nt)
    out_var = np.array([out_var[:, 0], out_var[:, 2]]).T  # divides the data column by the counter column
    return out_var

In [ ]:
if plotting==True:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    
    
    # List of profile arrays and their corresponding labels and x-axis titles
    profiles = [
        (ALL_profile_array_VMF_c.copy(), SHALLOW_profile_array_VMF_c.copy(), DEEP_profile_array_VMF_c.copy(), 'VMF_c (kg/m^2/s)'),
        (ALL_profile_array_VMF_g.copy(), SHALLOW_profile_array_VMF_g.copy(), DEEP_profile_array_VMF_g.copy(), 'VMF_g (kg/m^2/s)'),
    ]
    
    
    # Set up the 2x3 gridspec
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    
    # Loop through the profiles and plot them on subplots
    # for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
    for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
    
        ax = fig.add_subplot(gs[i])
     
        # Plot Profiles
        profile_list=[ALL_profile_array,SHALLOW_profile_array,DEEP_profile_array]
        colors=['black','green','blue']
        labels=['ALL','SHALLOW','DEEP']
        for (profile,color,label) in zip(profile_list,colors,labels):   
            profile=averaged_profiles(profile) # Compute the averaged profile
            
            ax.plot(profile[:, 0], profile[:, 1],color=color,label=label)
            
        ax.set_xlabel(xlabel)
        ax.set_ylabel('z (km)')
        ax.grid(True)
    
        #LEGEND
        if i==1:
            ax.legend(loc='upper center')
    
    
        apply_scientific_notation([ax])
    
    #MEAN CLOUD BASE
    axs = fig.get_axes()
    ax1 = axs[0]
    for axis in [ax1]:
        axis.axhline(all_cloudbase,color='purple',linestyle='dashed')
     
    #LEGEND
    # legend_ax = fig.add_subplot(gs[1, 2])  # Use the (2, 3) grid slot for the legend
    # legend_ax.axis("off")  # Hide axes for the legend box
    # legend_ax.legend(*ax.get_legend_handles_labels(), loc='center', frameon=False)
    
    #ACCESSORIES
    plt.suptitle('(ALL, SHALLOW < 4 km, DEEP > 6 km) Tracked Lagrangian CL (BL-LFC + 20 minutes) Parcels')
    plt.tight_layout()
    
    # #MATCHING X AXISES WITH NEXT PLOT
    # #(TO KEEP AXISES CONSISTENT RUN PLOTS IN ORDER: 1st Plots, 2nd Plots, 1st Plots)
    # for axis in axs:
    #     nbins=7;axis.xaxis.set_major_locator(MaxNLocator(nbins=nbins))
    # if 'ax8' in globals():
    #     fix_x_limits([ax1,ax8])
    #     fix_x_limits([ax2,ax9])
    #     fix_x_limits([ax3,ax10])
    #     fix_x_limits([ax4,ax11])
    #     fix_x_limits([ax5,ax12])
    #     fix_x_limits([ax6,ax13])
    #     fix_x_limits([ax7,ax14])
    #     del ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10,ax11,ax12,ax13,ax14


In [ ]:
if plotting==True:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    
    # List of profile arrays and their corresponding labels and x-axis titles
    profiles = [
        (nonCL_ALL_profile_array_VMF_c.copy(), nonCL_SHALLOW_profile_array_VMF_c.copy(), nonCL_DEEP_profile_array_VMF_c.copy(), 'VMF_c (kg/m^2/s)'),
        (nonCL_ALL_profile_array_VMF_g.copy(), nonCL_SHALLOW_profile_array_VMF_g.copy(), nonCL_DEEP_profile_array_VMF_g.copy(), 'VMF_g (kg/m^2/s)')
    ]
    
    # Set up the 2x3 gridspec
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    
    # Loop through the profiles and plot them on subplots
    # for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
    for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
    
        ax = fig.add_subplot(gs[i])
     
        # Plot Profiles
        profile_list=[ALL_profile_array,SHALLOW_profile_array,DEEP_profile_array]
        colors=['black','green','blue']
        labels=['ALL','SHALLOW','DEEP']
        for (profile,color,label) in zip(profile_list,colors,labels):   
            profile=averaged_profiles(profile) # Compute the averaged profile
            
    
            
            ax.plot(profile[:, 0], profile[:, 1],color=color,label=label)
            
        ax.set_xlabel(xlabel)
        ax.set_ylabel('z (km)')
        ax.grid(True)
    
        #LEGEND
        if i==1:
            ax.legend(loc='upper center')
    
    
        apply_scientific_notation([ax])
    
    #MEAN CLOUD BASE
    axs = fig.get_axes()
    ax1 = axs[0]
    for axis in [ax1]:
        axis.axhline(all_cloudbase,color='purple',linestyle='dashed')
        
    # #LEGEND
    # legend_ax = fig.add_subplot(gs[1, 2])  # Use the (2, 3) grid slot for the legend
    # legend_ax.axis("off")  # Hide axes for the legend box
    # legend_ax.legend(*ax.get_legend_handles_labels(), loc='center', frameon=False)
    
    #ACCESSORIES
    plt.suptitle('(ALL, SHALLOW < 4 km, DEEP > 6 km) Tracked Lagrangian nonCL (BL-LFC + 20 minutes) Parcels')
    plt.tight_layout()
    
    
    # # #FIXING X AXISES TO MATCH PREVIOUS PLOTS
    # for axis in axs:
    #     nbins=7;axis.xaxis.set_major_locator(MaxNLocator(nbins=nbins))
    # fix_x_limits([ax1,ax8])
    # fix_x_limits([ax2,ax9])
    # fix_x_limits([ax3,ax10])
    # fix_x_limits([ax4,ax11])
    # fix_x_limits([ax5,ax12])
    # fix_x_limits([ax6,ax13])
    # fix_x_limits([ax7,ax14])

In [ ]:
if plotting==True:
    #DIFFERENCES
    
    def average_difference(array1, array2):
        out_var_one=averaged_profiles(array1)
        out_var_two=averaged_profiles(array2)
    
        #masking out non matches
        second_col_one = out_var_one[:, 1]
        second_col_two = out_var_two[:, 1]
        mask_one = np.isin(second_col_one, second_col_two)
        mask_two = np.isin(second_col_two, second_col_one)
        
        out_var_one = out_var_one.copy()[mask_one]
        out_var_two = out_var_two.copy()[mask_two]
        
        diff=(out_var_one[:,0]-out_var_two[:,0])
        zs=out_var_one[:,1]
    
        out_profile=np.zeros((len(diff),2))
    
        out_profile[:,0]=diff;out_profile[:,1]=zs;
        return out_profile
    
    # Calculate average differences for each variable (w, qv, qc, th, buoyancy)
    ALL_diff_VMF_c = average_difference(ALL_profile_array_VMF_c, nonCL_ALL_profile_array_VMF_c)
    SHALLOW_diff_VMF_c = average_difference(SHALLOW_profile_array_VMF_c, nonCL_SHALLOW_profile_array_VMF_c)
    DEEP_diff_VMF_c= average_difference(DEEP_profile_array_VMF_c, nonCL_DEEP_profile_array_VMF_c)
    
    ALL_diff_VMF_g = average_difference(ALL_profile_array_VMF_g, nonCL_ALL_profile_array_VMF_g)
    SHALLOW_diff_VMF_g = average_difference(SHALLOW_profile_array_VMF_g, nonCL_SHALLOW_profile_array_VMF_g)
    DEEP_diff_VMF_g= average_difference(DEEP_profile_array_VMF_g, nonCL_DEEP_profile_array_VMF_g)
    print('done')

In [ ]:
if plotting==True:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    
    # List of profile arrays and their corresponding labels and x-axis titles
    profiles = [
        (ALL_diff_VMF_c.copy(), SHALLOW_diff_VMF_c.copy(), DEEP_diff_VMF_c.copy(), 'VMF_c (kg/m^2/s)'),
        (ALL_diff_VMF_g.copy(), SHALLOW_diff_VMF_g.copy(), DEEP_diff_VMF_g.copy(), 'VMF_g (kg/m^2/s)')
    ]
    
    # Set up the 2x3 gridspec
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    
    # Loop through the profiles and plot them on subplots
    for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
        ax = fig.add_subplot(gs[i])
    
        
        # Compute the averaged profile
        ALL_profile = ALL_profile_array.copy()
        SHALLOW_profile = SHALLOW_profile_array.copy()
        DEEP_profile = DEEP_profile_array.copy()
    
        if xlabel in [r'$q_v$ (g/kg)',r'$q_c+q_i$ (g/kg)','HMC (g/kg/s)']:
            ALL_profile[:,0]*=1000
            SHALLOW_profile[:,0]*=1000
            DEEP_profile[:,0]*=1000
        
        # Plot the profile
        ax.plot(ALL_profile[:, 0], ALL_profile[:, 1],color='black',label='ALL CL-NonCL')
        ax.plot(SHALLOW_profile[:, 0], SHALLOW_profile[:, 1],color='green',label='SHALLOW CL-NonCL')
        ax.plot(DEEP_profile[:, 0], DEEP_profile[:, 1],color='blue',label='DEEP CL-NonCL')
        ax.set_xlabel(xlabel)
        ax.set_ylabel('z (km)')
        ax.grid(True)
    
        ax.axvline(0,color='k',linestyle='dashed')
    
        #LEGEND
        if i==1:
            ax.legend(loc='upper left')
            
        apply_scientific_notation([ax])
    
    
    #MEAN CLOUD BASE
    axs = fig.get_axes()
    ax1 = axs[0]
    for axis in [ax1]:
        axis.axhline(all_cloudbase,color='purple',linestyle='dashed')
        
    # #LEGEND
    # legend_ax = fig.add_subplot(gs[1, 2])  # Use the (2, 3) grid slot for the legend
    # legend_ax.axis("off")  # Hide axes for the legend box
    # legend_ax.legend(*ax.get_legend_handles_labels(), loc='center', frameon=False)
    
    #ACCESSORIES
    plt.suptitle('(ALL, SHALLOW < 4 km, DEEP > 6 km) Tracked Lagrangian CL-nonCL Difference (BL-LFC + 30 minutes) Parcels')
    plt.tight_layout()
    


In [ ]:
#SBZ vs nonSBZ
################################################################################

In [ ]:
#FINDING SBZ CONVERGENCE

#FIND ALL X MAXES
import sys
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir+'../Functions'
sys.path.append(path)
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

# TF=False
# if TF==False:
#     print('Loading In Data')
#     u_data=data['u'].interp(xf=data['xh']).data
#     v_data=data['v'].interp(yf=data['yh']).data
#     TF=True


from NumericalFunctions import *
# def find_SBZ_xmaxs():
    
#     # print('calculating convergence and taking mean')
#     # Conv=-(Ddx(u_data,1000)+Ddy(v_data,1000))
#     dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
#     file_path = dir2 + 'Variable_Calculation/' + 'Convergence' + f'_{res}_5min' + '.h5'
#     with h5py.File(file_path, 'r') as f:
#         Conv = f['conv'][:]
    
#     zlev=4
#     Conv_ymean=np.mean(Conv[:,zlev],axis=1)
#     xmaxs=np.argmax(Conv_ymean, axis=1)

#     return xmaxs
def find_SBZ_xmaxs():
    # Define the directory and file path
    dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
    file_path = dir2 + 'Variable_Calculation/' + 'Convergence' + f'_{res}_{Np_str}_5min' + '.h5'
    
    # Open the HDF5 file in read mode
    with h5py.File(file_path, 'r') as f:
        # Access the 'conv' dataset
        conv_dataset = f['conv']
        
        # Define the vertical level you are interested in
        zlev = 4
        
        # Initialize a list to store the xmaxs for each time step
        xmaxs_list = []

        # Loop over each time step (axis=0 corresponds to time)
        for t in range(conv_dataset.shape[0]):  # conv_dataset.shape[0] is the time dimension size
            # Read the relevant slice for this time step and vertical level
            Conv_t_zlev = conv_dataset[t, zlev, :, :]  # Shape should be (y_size, x_size)
            
            # Calculate the mean across the y-axis
            Conv_ymean = np.mean(Conv_t_zlev, axis=0)  # Mean across the y-axis
            
            # Find the index of the maximum value along the x-axis
            xmax = np.argmax(Conv_ymean)
            
            # Append the result for this time step
            xmaxs_list.append(xmax)
    
    # Convert the list of xmaxs to a numpy array (optional)
    xmaxs = np.array(xmaxs_list)

    return xmaxs #returns SBZ x location for each timestep


def subset_SBZ(out_arr):
    xmaxs=find_SBZ_xmaxs()

    SBZ_subset=[]
    # test=[] #TESTING
    
    for ind in np.arange(out_arr.shape[0]):
        
        row=out_arr[ind]
        p=row[0]
        t=row[1]

        kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1)
        if X[t,p-index_adjust] in np.arange( (xmaxs[t]-2*kms),(xmaxs[t]+2*kms) +1):
            SBZ_subset.append(ind)
            # test.append(p) #TESTING
    
    SBZ_out_arr=out_arr[SBZ_subset]
    print(f'there are a total of {len(SBZ_out_arr)} ALL SBZ CL parcels')

    valid_range=np.arange(out_arr.shape[0])
    nonSBZ_out_arr=out_arr[list(set(valid_range) - set(SBZ_subset))]
    print(f'there are a total of {len(nonSBZ_out_arr)} ALL nonSBZ CL parcels')
    return SBZ_out_arr,nonSBZ_out_arr


# #LOADING CL MAXS FROM CL TRACKING ALGORITHM
# folder = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/'
# whereSBZ=xr.open_dataset(folder+f'whereCL_{res}_ONLY_SBZS.nc').load()
# whereSBZ=whereSBZ.isel(time=slice(0,len(data['time'])))
# whereSBZ=whereSBZ['maxconv_x']
# def Get_SBZ_X(t,z,y):
#     Conv_X_Max=whereSBZ[t,z,y,:].values
#     return Conv_X_Max
# def subset_SBZ(out_arr):

#     SBZ_subset=[]
#     # test=[] #TESTING
    
#     for ind in np.arange(out_arr.shape[0]):
        
#         row=out_arr[ind]
#         p=row[0]
#         t=row[1]

#         kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1)
#         value=X[t,p]
#         if np.any((value >= xmaxs - 2*kms) & (value <= xmaxs + 2*kms))==True:
#             SBZ_subset.append(ind)
#             # test.append(p) #TESTING
    
#     SBZ_out_arr=out_arr[SBZ_subset]
#     print(f'there are a total of {len(SBZ_out_arr)} ALL SBZ CL parcels')

#     valid_range=np.arange(out_arr.shape[0])
#     nonSBZ_out_arr=out_arr[list(set(valid_range) - set(SBZ_subset))]
#     print(f'there are a total of {len(nonSBZ_out_arr)} ALL nonSBZ CL parcels')
#     return SBZ_out_arr,nonSBZ_out_arr

#SUBSETTING OUT SHALLOW AND DEEP FROM SBZ AND NONSBZ
def SHALLOW_threshold(zthresh,type):
    if type=='SBZ':
        out_arr=ALL_SBZ_out_arr.copy()
    elif type=='nonSBZ':
        out_arr=ALL_nonSBZ_out_arr.copy()
    
    deep_out_ind=[]; extendrange=[]
    times=data['time'].values/(1e9 * 60); times=times.astype(float);
    for ind in range(len(out_arr)): 
        # if np.mod(ind,5000)==0: print(f'{ind}/{len(out_arr)}')
        #CHECK TO SSEE IF NEXT MOST LOCAL TIME MAX GOES ABOVE ZTHRESHS 

        #Get Ascending Range Past LFC For Maximum 120 Minutes Simulation Time
        nummins=120; numsteps=int(nummins/times[1])
        aboverange=np.arange(out_arr[ind,2],out_arr[ind,2]+numsteps,1) #range of times between current time and numsteps later
        aboverange=aboverange[aboverange<len(data['time'])] #caps out at max time
        above=parcel_z[aboverange,out_arr[ind,0]-index_adjust]/1000 #JOBARRAY INDEXADJUST
        
        #Takes The time derivative 
        ddx=ddt(above)

        #Checks whether the Local Time Max Is Located Above zthresh
        signs = np.sign(ddx)
        signs_diff=np.diff(signs)
        local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here
        if len(local_maxes)==0:
            local_maxes=[0]
        
        if np.any(above[local_maxes[0]]<=zthresh): #< for SHALLOW, > for DEEP
            extendrange.append(local_maxes[0]) #save to extend xlim of plot later
            deep_out_ind.append(ind)

    #SUBSET OUT FOR FINAL RESULT
    out_arr=out_arr[deep_out_ind,:]
    # print(f'> {zthresh} km. {len(out_arr)} leftover parcels')
    return out_arr#, extendrange
    # print(out_arr)

def DEEP_threshold(zthresh,type):
    if type=='SBZ':
        out_arr=ALL_SBZ_out_arr.copy()
    elif type=='nonSBZ':
        out_arr=ALL_nonSBZ_out_arr.copy()

    deep_out_ind=[]; extendrange=[]
    times=data['time'].values/(1e9 * 60); times=times.astype(float);
    for ind in range(len(out_arr)): 
        # if np.mod(ind,5000)==0: print(f'{ind}/{len(out_arr)}')
        #CHECK TO SSEE IF NEXT MOST LOCAL TIME MAX GOES ABOVE ZTHRESHS 
        
        #Get Ascending Range Past LFC For Maximum 120 Minutes Simulation Time
        nummins=120; numsteps=int(nummins/times[1])
        aboverange=np.arange(out_arr[ind,2],out_arr[ind,2]+numsteps,1) #range of times between current time and numsteps later
        aboverange=aboverange[aboverange<len(data['time'])] #caps out at max time
        above=parcel_z[aboverange,out_arr[ind,0]-index_adjust]/1000 #JOBARRAY INDEX_ADJUST
    
        #Takes The time derivative 
        ddx=ddt(above)

        #Checks whether the Local Time Max Is Located Above zthresh
        signs = np.sign(ddx)
        signs_diff=np.diff(signs)
        local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here
        if len(local_maxes)==0:
            local_maxes=[0]
        
        if np.any(above[local_maxes[0]]>=zthresh): #< for SHALLOW, > for DEEP
            extendrange.append(local_maxes[0]) #save to extend xlim of plot later
            deep_out_ind.append(ind)

    #SUBSET OUT FOR FINAL RESULT
    out_arr=out_arr[deep_out_ind,:]
    # print(f'> {zthresh} km. {len(out_arr)} leftover parcels')
    return out_arr#, extendrange
    # print(out_arr)
##############################################################
[ALL_SBZ_out_arr,ALL_nonSBZ_out_arr]=subset_SBZ(ALL_out_arr)
SHALLOW_SBZ_out_arr=SHALLOW_threshold(4,'SBZ')
print(f'there are a total of {len(SHALLOW_SBZ_out_arr)} SHALLOW SBZ CL parcels')
SHALLOW_nonSBZ_out_arr=SHALLOW_threshold(4,'nonSBZ')
print(f'there are a total of {len(SHALLOW_nonSBZ_out_arr)} SHALLOW nonSBZ CL parcels')
DEEP_SBZ_out_arr=DEEP_threshold(6,'SBZ')
print(f'there are a total of {len(DEEP_SBZ_out_arr)} DEEP SBZ CL parcels')
DEEP_nonSBZ_out_arr=DEEP_threshold(6,'nonSBZ')
print(f'there are a total of {len(DEEP_nonSBZ_out_arr)} DEEP nonSBZ CL parcels')

In [ ]:
#SBZ After Arrays
ALL_SBZ_out_after_array=find_after_time(ALL_SBZ_out_arr)
SHALLOW_SBZ_out_after_array=np.zeros(len(SHALLOW_SBZ_out_arr),dtype=int)
DEEP_SBZ_out_after_array=find_after_time(DEEP_SBZ_out_arr)

ALL_nonSBZ_out_after_array=find_after_time(ALL_nonSBZ_out_arr)
SHALLOW_nonSBZ_out_after_array=np.zeros(len(SHALLOW_nonSBZ_out_arr),dtype=int)
DEEP_nonSBZ_out_after_array=find_after_time(DEEP_nonSBZ_out_arr)

In [ ]:
#FINDING MEAN CLOUD BASE #*****************************
zh=data['zh'].values
w_thresh2=0.5
qcqi_thresh=1e-6
type='all'

if type=='all':
    out_arr=ALL_out_arr.copy()
elif type=='deep':
    out_arr=DEEP_out_arr.copy()
elif type=='shallow':
    out_arr=SHALLOW_out_arr.copy()

zhs=data['zh'].values
profile_array =np.zeros((len(zhs), 2)) #column 1: var, column 2: counter, column 3: list of zhs
profile_array[:,1]=zhs;

# cloudbase_lst=[]
after=20*int(minutes) #20 minutes
for row in range(out_arr.shape[0]):
    if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
    p=out_arr[row,0]
    
    # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
    ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
    ts = np.arange(out_arr[row, 1], ts_end)
    
    zs=Z[ts,p-index_adjust]
    ys=Y[ts,p-index_adjust]
    xs=X[ts,p-index_adjust]

    ws=W[ts,p-index_adjust]
    qcqis=QCQI[ts,p-index_adjust]
    where=np.where((ws>=w_thresh2) & (qcqis>=qcqi_thresh))
    profile_array[zs[where],0]+=1
del after
# print(np.mean(cloudbase_lst))
# print(np.min(cloudbase_lst))
# plt.hist(cloudbase_lst,bins=40,orientation='horizontal');
all_cloudbase=zh[np.where(profile_array[:,0]!=0)[0][0]]
print(all_cloudbase)

In [ ]:
#PLOTTING SBZ vs non-SBZ Vertical Profiles
##########################################

#SBZ
def SBZ_tracked_profile(var_data,type):

    if type=='all':
        out_arr=ALL_SBZ_out_arr.copy()
        after_array=ALL_SBZ_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_SBZ_out_arr.copy()
        after_array=SHALLOW_SBZ_out_after_array
    elif type=='deep':
        out_arr=DEEP_SBZ_out_arr.copy()
        after_array=DEEP_SBZ_out_after_array
    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=after_array[row]
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        vars=var_data[ts,p-index_adjust]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array


print('ALL')
ALL_profile_array_VMF_c=SBZ_tracked_profile(VMF_c,type='all')
ALL_profile_array_VMF_g=SBZ_tracked_profile(VMF_g,type='all')

print('SHALLOW')
SHALLOW_profile_array_VMF_c=SBZ_tracked_profile(VMF_c,type='shallow')
SHALLOW_profile_array_VMF_g=SBZ_tracked_profile(VMF_g,type='shallow')

print('DEEP')
DEEP_profile_array_VMF_c=SBZ_tracked_profile(VMF_c,type='deep')
DEEP_profile_array_VMF_g=SBZ_tracked_profile(VMF_g,type='deep')


In [ ]:
#nonSBZ
def nonSBZ_tracked_profile(var_data,type):
    if type=='all':
        out_arr=ALL_nonSBZ_out_arr.copy()
        after_array=ALL_nonSBZ_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_nonSBZ_out_arr.copy()
        after_array=SHALLOW_nonSBZ_out_after_array
    elif type=='deep':
        out_arr=DEEP_nonSBZ_out_arr.copy()
        after_array=DEEP_nonSBZ_out_after_array  

    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=after_array[row]
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        vars=var_data[ts,p-index_adjust]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array


print('ALL')
nonSBZ_ALL_profile_array_VMF_c=nonSBZ_tracked_profile(VMF_c,type='all')
nonSBZ_ALL_profile_array_VMF_g=nonSBZ_tracked_profile(VMF_g,type='all')

print('DEEP')
nonSBZ_DEEP_profile_array_VMF_c=nonSBZ_tracked_profile(VMF_c,type='deep')
nonSBZ_DEEP_profile_array_VMF_g=nonSBZ_tracked_profile(VMF_g,type='deep')

print('SHALLOW')
nonSBZ_SHALLOW_profile_array_VMF_c=nonSBZ_tracked_profile(VMF_c,type='shallow')
nonSBZ_SHALLOW_profile_array_VMF_g=nonSBZ_tracked_profile(VMF_g,type='shallow')

In [ ]:
#SAVING

# Define categories and variables
types = ["ALL", "SHALLOW", "DEEP"]
variables = ["VMF_c","VMF_g"]

# Dictionary to store all variables
tracked_profiles = {}

# Populate all profiles in one loop
for type in types:
    for var in variables:
        tracked_profiles[f"{type}_profile_array_{var}"] = eval(f"{type}_profile_array_{var}")
        tracked_profiles[f"nonSBZ_{type}_profile_array_{var}"] = eval(f"nonSBZ_{type}_profile_array_{var}")


# Save all variables in an HDF5 file
dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out2/'
output_file=dir2+f"SBZ_nonSBZ_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min_{job_id}.h5"
with h5py.File(output_file, "w") as h5f:
    for name, profile_data in tracked_profiles.items():
        h5f.create_dataset(name, data=profile_data)
print('done')


In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [ ]:
if recombine==True:
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["VMF_c"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_profile_array_{var}")

In [ ]:
if recombine==True:
    dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out2/'
    
    
    #MAKING OUTPUT FILE PATH
    output_file=dir2+f"SBZ_nonSBZ_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min.h5"
    
    #MAKING PROFILES DICTIONARY
    zhs = data['zh'].values
    profiles = {}  # Store profiles for all variables
    for var in vars_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs 

    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
    
        #CALLING IN DATA
        input_file=dir2+f"SBZ_nonSBZ_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min_{job_id}.h5"
    
        #COMPILING PROFILES
        with h5py.File(input_file, 'r') as f:
            for var in vars_list:
                profiles[var][:,0:1+1]+=f[f'{var}'][:,0:1+1]
    
    #SAVING INTO FINAL FORM
    with h5py.File(output_file, 'w') as f:
        for var in profiles:
            profile_var = profiles[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

In [ ]:
##############################
#PLOTTING
plotting=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# plotting=True

In [ ]:
if plotting==True:
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["VMF_c","VMF_g"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"nonSBZ_{t}_profile_array_{var}")
    
    # Define directory and output file path
    dir2 = dir + 'Project_Algorithms/Tracked_Profiles/job_out2/'
    output_file = dir2+f"SBZ_nonSBZ_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min.h5"
    
    # Open the HDF5 file and read the stored datasets into dynamically named variables
    with h5py.File(output_file, 'r') as f:
        for var in vars_list:
            globals()[var] = f[f'{var}'][:]


In [ ]:
if plotting==True:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    
    
    # List of profile arrays and their corresponding labels and x-axis titles
    profiles = [
        (ALL_profile_array_VMF_c.copy(), SHALLOW_profile_array_VMF_c.copy(), DEEP_profile_array_VMF_c.copy(), 'VMF_c (kg/m^2/s)'),
        (ALL_profile_array_VMF_g.copy(), SHALLOW_profile_array_VMF_g.copy(), DEEP_profile_array_VMF_g.copy(), 'VMF_g (kg/m^2/s)')
    ]
    
    
    # Set up the 2x3 gridspec
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    
    # Loop through the profiles and plot them on subplots
    # for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
    for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
    
        ax = fig.add_subplot(gs[i])
     
        # Plot Profiles
        profile_list=[ALL_profile_array,SHALLOW_profile_array,DEEP_profile_array]
        colors=['black','green','blue']
        labels=['ALL','SHALLOW','DEEP']
        for (profile,color,label) in zip(profile_list,colors,labels):   
            profile=averaged_profiles(profile) # Compute the averaged profile
            
            
            ax.plot(profile[:, 0], profile[:, 1],color=color,label=label)
            
        ax.set_xlabel(xlabel)
        ax.set_ylabel('z (km)')
        ax.grid(True)
    
        #LEGEND
        if i==1:
            ax.legend(loc='upper center')
    
        apply_scientific_notation([ax])
    
    #MEAN CLOUD BASE
    axs = fig.get_axes()
    ax1 = axs[0]
    for axis in [ax1]:
        axis.axhline(all_cloudbase,color='purple',linestyle='dashed')
     
     
    #LEGEND
    # legend_ax = fig.add_subplot(gs[1, 2])  # Use the (2, 3) grid slot for the legend
    # legend_ax.axis("off")  # Hide axes for the legend box
    # legend_ax.legend(*ax.get_legend_handles_labels(), loc='center', frameon=False)
    
    #ACCESSORIES
    plt.suptitle('(ALL, SHALLOW < 4 km, DEEP > 6 km) Tracked Lagrangian SBZ (BL-LFC + 20 minutes) Parcels')
    plt.tight_layout()
    
    # #MATCHING X AXISES WITH NEXT PLOT
    # #(TO KEEP AXISES CONSISTENT RUN PLOTS IN ORDER: 1st Plots, 2nd Plots, 1st Plots)
    # for axis in axs:
    #     nbins=7;axis.xaxis.set_major_locator(MaxNLocator(nbins=nbins))
    # if 'ax8' in globals():
    #     fix_x_limits([ax1,ax8])
    #     fix_x_limits([ax2,ax9])
    #     fix_x_limits([ax3,ax10])
    #     fix_x_limits([ax4,ax11])
    #     fix_x_limits([ax5,ax12])
    #     fix_x_limits([ax6,ax13])
    #     fix_x_limits([ax7,ax14])
    #     del ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10,ax11,ax12,ax13,ax14


In [ ]:
if plotting==True:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    
    # List of profile arrays and their corresponding labels and x-axis titles
    profiles = [
        (nonSBZ_ALL_profile_array_VMF_c.copy(), nonSBZ_SHALLOW_profile_array_VMF_c.copy(), nonSBZ_DEEP_profile_array_VMF_c.copy(), 'VMF_c (kg/m^2/s)'),
        (nonSBZ_ALL_profile_array_VMF_g.copy(), nonSBZ_SHALLOW_profile_array_VMF_g.copy(), nonSBZ_DEEP_profile_array_VMF_g.copy(), 'VMF_g (kg/m^2/s)')
    ]
    
    
    # Set up the 2x3 gridspec
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    
    # Loop through the profiles and plot them on subplots
    # for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
    for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
    
        ax = fig.add_subplot(gs[i])
     
        # Plot Profiles
        profile_list=[ALL_profile_array,SHALLOW_profile_array,DEEP_profile_array]
        colors=['black','green','blue']
        labels=['ALL','SHALLOW','DEEP']
        for (profile,color,label) in zip(profile_list,colors,labels):   
            profile=averaged_profiles(profile) # Compute the averaged profile
            
    
            ax.plot(profile[:, 0], profile[:, 1],color=color,label=label)
            
        ax.set_xlabel(xlabel)
        ax.set_ylabel('z (km)')
        ax.grid(True)
    
        #LEGEND
        if i==1:
            ax.legend(loc='upper center')
    
        apply_scientific_notation([ax])
    
    #MEAN CLOUD BASE
    axs = fig.get_axes()
    ax1 = axs[0]
    for axis in [ax1]:
        axis.axhline(all_cloudbase,color='purple',linestyle='dashed')
        
    # # #LEGEND
    # # legend_ax = fig.add_subplot(gs[1, 2])  # Use the (2, 3) grid slot for the legend
    # # legend_ax.axis("off")  # Hide axes for the legend box
    # # legend_ax.legend(*ax.get_legend_handles_labels(), loc='center', frameon=False)
    
    # #ACCESSORIES
    # plt.suptitle('(ALL, SHALLOW < 4 km, DEEP > 6 km) Tracked Lagrangian nonSBZ (BL-LFC + 20 minutes) Parcels')
    # plt.tight_layout()
    
    
    # # #FIXING X AXISES TO MATCH PREVIOUS PLOTS
    # for axis in axs:
    #     nbins=7;axis.xaxis.set_major_locator(MaxNLocator(nbins=nbins))
    # fix_x_limits([ax1,ax8])
    # fix_x_limits([ax2,ax9])
    # fix_x_limits([ax3,ax10])
    # fix_x_limits([ax4,ax11])
    # fix_x_limits([ax5,ax12])
    # fix_x_limits([ax6,ax13])
    # fix_x_limits([ax7,ax14])

In [ ]:
if plotting==True:
    #SBZ vs nonSBZ
    def average_difference(array1, array2):
        out_var_one=averaged_profiles(array1)
        out_var_two=averaged_profiles(array2)
    
        #masking out non matches
        second_col_one = out_var_one[:, 1]
        second_col_two = out_var_two[:, 1]
        mask_one = np.isin(second_col_one, second_col_two)
        mask_two = np.isin(second_col_two, second_col_one)
        
        out_var_one = out_var_one.copy()[mask_one]
        out_var_two = out_var_two.copy()[mask_two]
        
        diff=(out_var_one[:,0]-out_var_two[:,0])
        zs=out_var_one[:,1]
    
        out_profile=np.zeros((len(diff),2))
    
        out_profile[:,0]=diff;out_profile[:,1]=zs;
        return out_profile
            
    # Calculate average differences for each variable (w, qv, qc, th, buoyancy) for SBZ
    print('VMF_c')
    ALL_diff_VMF_c = average_difference(ALL_profile_array_VMF_c, nonSBZ_ALL_profile_array_VMF_c)
    SHALLOW_diff_VMF_c = average_difference(SHALLOW_profile_array_VMF_c, nonSBZ_SHALLOW_profile_array_VMF_c)
    DEEP_diff_VMF_c = average_difference(DEEP_profile_array_VMF_c, nonSBZ_DEEP_profile_array_VMF_c)
    
    print('VMF_g')
    ALL_diff_VMF_g = average_difference(ALL_profile_array_VMF_g, nonSBZ_ALL_profile_array_VMF_g)
    SHALLOW_diff_VMF_g = average_difference(SHALLOW_profile_array_VMF_g, nonSBZ_SHALLOW_profile_array_VMF_g)
    DEEP_diff_VMF_g = average_difference(DEEP_profile_array_VMF_g, nonSBZ_DEEP_profile_array_VMF_g)

In [ ]:
if plotting==True:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    
    # List of profile arrays and their corresponding labels and x-axis titles
    profiles = [
        (ALL_diff_VMF_c.copy(), SHALLOW_diff_VMF_c.copy(), DEEP_diff_VMF_c.copy(), 'VMF_c (kg/m^2/s)'),
        (ALL_diff_VMF_g.copy(), SHALLOW_diff_VMF_g.copy(), DEEP_diff_VMF_g.copy(), 'VMF_g (kg/m^2/s)')
    ]
    
    
    # Set up the 2x3 gridspec
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    
    # Loop through the profiles and plot them on subplots
    for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
        ax = fig.add_subplot(gs[i])
    
        
        # Compute the averaged profile
        ALL_profile = ALL_profile_array.copy()
        SHALLOW_profile = SHALLOW_profile_array.copy()
        DEEP_profile = DEEP_profile_array.copy()
    
        if xlabel in [r'$q_v$ (g/kg)',r'$q_c+q_i$ (g/kg)','HMC (g/kg/s)']:
            ALL_profile[:,0]*=1000
            SHALLOW_profile[:,0]*=1000
            DEEP_profile[:,0]*=1000
        
        # Plot the profile
        ax.plot(ALL_profile[:, 0], ALL_profile[:, 1],color='black',label='ALL CL-NonCL')
        ax.plot(SHALLOW_profile[:, 0], SHALLOW_profile[:, 1],color='green',label='SHALLOW CL-NonCL')
        ax.plot(DEEP_profile[:, 0], DEEP_profile[:, 1],color='blue',label='DEEP CL-NonCL')
        ax.set_xlabel(xlabel)
        ax.set_ylabel('z (km)')
        ax.grid(True)
    
        ax.axvline(0,color='k',linestyle='dashed')
    
        #LEGEND
        if i==1:
            ax.legend(loc='upper left')
            
        apply_scientific_notation([ax])
    
    #MEAN CLOUD BASE
    axs = fig.get_axes()
    ax1=axs[0]
    for axis in [ax1]:
        axis.axhline(all_cloudbase,color='purple',linestyle='dashed')
        
    # #LEGEND
    # legend_ax = fig.add_subplot(gs[1, 2])  # Use the (2, 3) grid slot for the legend
    # legend_ax.axis("off")  # Hide axes for the legend box
    # legend_ax.legend(*ax.get_legend_handles_labels(), loc='center', frameon=False)
    
    #ACCESSORIES
    plt.suptitle('(ALL, SHALLOW < 4 km, DEEP > 6 km) Tracked Lagrangian SBZ-nonSBZ Difference (BL-LFC + 30 minutes) Parcels')
    plt.tight_layout()
    


In [ ]:
#ColdPool
################################################################

In [ ]:
#COLD POOL
def get_ColdPool(out_arr1,out_arr2):
    arr1 = out_arr1[:,0] #CL
    arr2 = out_arr2[:,0] #nonSBZ
    common_values = np.intersect1d(arr1, arr2)
    indices_arr1 = np.where(np.isin(arr1, common_values))[0]  # Indices in arr1
    ColdPool_out_arr=out_arr1[indices_arr1]
    return ColdPool_out_arr

#ALL
ALL_ColdPool_out_arr = get_ColdPool(ALL_out_arr, ALL_nonSBZ_out_arr)
ALL_ColdPool_after_array=find_after_time(ALL_ColdPool_out_arr)

#SHALLOW
SHALLOW_ColdPool_out_arr = get_ColdPool(SHALLOW_out_arr, SHALLOW_nonSBZ_out_arr)
SHALLOW_ColdPool_after_array=np.zeros(len(ALL_ColdPool_after_array),dtype=int)

#DEEP
DEEP_ColdPool_out_arr = get_ColdPool(DEEP_out_arr, DEEP_nonSBZ_out_arr)
DEEP_ColdPool_after_array=find_after_time(DEEP_ColdPool_out_arr)

In [ ]:
#PLOTTING SBZ vs non-SBZ Vertical Profiles
##########################################

#SBZ
def ColdPool_tracked_profile(var_data,type):

    if type=='all':
        out_arr=ALL_ColdPool_out_arr.copy()
        after_array=ALL_ColdPool_after_array
    elif type=='shallow':
        out_arr=SHALLOW_ColdPool_out_arr
        after_array=SHALLOW_ColdPool_after_array
    elif type=='deep':
        out_arr=DEEP_ColdPool_out_arr
        after_array=DEEP_ColdPool_after_array
    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=after_array[row]
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        vars=var_data[ts,p-index_adjust]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array

print('ALL')
ALL_ColdPool_profile_array_VMF_c = ColdPool_tracked_profile(VMF_c, type='all')
ALL_ColdPool_profile_array_VMF_g = ColdPool_tracked_profile(VMF_g, type='all')

print('DEEP')
DEEP_ColdPool_profile_array_VMF_c = ColdPool_tracked_profile(VMF_c, type='deep')
DEEP_ColdPool_profile_array_VMF_g = ColdPool_tracked_profile(VMF_g, type='deep')

print('SHALLOW')
SHALLOW_ColdPool_profile_array_VMF_c = ColdPool_tracked_profile(VMF_c, type='shallow')
SHALLOW_ColdPool_profile_array_VMF_g = ColdPool_tracked_profile(VMF_g, type='shallow')

In [ ]:
#SAVING

# Define categories and variables
types = ["ALL", "SHALLOW", "DEEP"]
variables = ["VMF_c","VMF_g"]

# Dictionary to store all variables
tracked_profiles = {}

# Populate all profiles in one loop
for type in types:
    for var in variables:
        tracked_profiles[f"{type}_ColdPool_profile_array_{var}"] = eval(f"{type}_ColdPool_profile_array_{var}")
        
# Save all variables in an HDF5 file
dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out2/'
output_file=dir2+f"ColdPool_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min_{job_id}.h5"
with h5py.File(output_file, "w") as h5f:
    for name, profile_data in tracked_profiles.items():
        h5f.create_dataset(name, data=profile_data)
print('done')


In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [ ]:
recombine=True:
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["w", "qv", "qcqi", "th", "th_e", "buoyancy", "HMC"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_ColdPool_profile_array_{var}")
    for t in types:
        for var in variables:
            SE_list.append(f"{t}_ColdPool_SE_{var}")

In [ ]:
recombine=True:
    dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out2/'
    
    #MAKING OUTPUT FILE PATH
    output_file=dir2+f"ColdPool_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min.h5"
    
    #MAKING PROFILES DICTIONARY
    zhs = data['zh'].values
    profiles = {}  # Store profiles for all variables
    for var in vars_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs 
    for var in SE_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs 
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
    
        #CALLING IN DATA
        input_file=dir2+f"ColdPool_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min_{job_id}.h5"
    
        #COMPILING PROFILES
        with h5py.File(input_file, 'r') as f:
            for var in vars_list + SE_list:  
                profiles[var][:,0:1+1]+=f[f'{var}'][:,0:1+1]
    
    #SAVING INTO FINAL FORM
    with h5py.File(output_file, 'w') as f:
        for var in profiles:
            profile_var = profiles[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

In [ ]:
##############################
#PLOTTING
plotting=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
plotting=True

In [ ]:
if plotting==True:
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["VMF_c","VMF_g"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_profile_array_{var}")
            vars_list.append(f"nonSBZ_{t}_profile_array_{var}")
    
    # Define directory and output file path
    dir2 = dir + 'Project_Algorithms/Tracked_Profiles/job_out2/'
    output_file = dir2 + f"SBZ_nonSBZ_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min.h5"
    
    # Open the HDF5 file and read the stored datasets into dynamically named variables
    with h5py.File(output_file, 'r') as f:
        for var in vars_list:
            globals()[var] = f[f'{var}'][:]

In [ ]:
if plotting==True:
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["VMF_c",'VMF_g']
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_ColdPool_profile_array_{var}")
    
    # Define directory and output file path
    dir2 = dir + 'Project_Algorithms/Tracked_Profiles/job_out2/'
    input_file = dir2 + f"ColdPool_tracked_eulerian_VMF_profiles_{res}_{Np_str}_5min.h5"
    
    # Open the HDF5 file and read the stored datasets into dynamically named variables
    with h5py.File(input_file, 'r') as f:
        for var in vars_list:
            globals()[var] = f[f'{var}'][:]


In [ ]:
if plotting==True:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    
    
    # List of profile arrays and their corresponding labels and x-axis titles
    profiles = [
        (ALL_profile_array_VMF_c.copy(), SHALLOW_profile_array_VMF_c.copy(), DEEP_profile_array_VMF_c.copy(), 'VMF_c (kg/m^2/s)'),
        (ALL_profile_array_VMF_g.copy(), SHALLOW_profile_array_VMF_g.copy(), DEEP_profile_array_VMF_g.copy(), 'VMF_g (kg/m^2/s)')
    ]
    
    
    
    # Set up the 2x3 gridspec
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    
    # Loop through the profiles and plot them on subplots
    # for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
    for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
    
        ax = fig.add_subplot(gs[i])
     
        # Plot Profiles
        profile_list=[ALL_profile_array,SHALLOW_profile_array,DEEP_profile_array]
        colors=['black','green','blue']
        labels=['ALL','SHALLOW','DEEP']
        for (profile,color,label) in zip(profile_list,colors,labels):   
            profile=averaged_profiles(profile) # Compute the averaged profile
            
                    
            ax.plot(profile[:, 0], profile[:, 1],color=color,label=label)
            
        ax.set_xlabel(xlabel)
        ax.set_ylabel('z (km)')
        ax.grid(True)
    
        #LEGEND
        if i==1:
            ax.legend(loc='upper center')
    
        apply_scientific_notation([ax])
    
    #MEAN CLOUD BASE
    axs = fig.get_axes()
    ax1=axs[0]
    for axis in [ax1]:
        axis.axhline(all_cloudbase,color='purple',linestyle='dashed')
     
    # #LEGEND
    # # legend_ax = fig.add_subplot(gs[1, 2])  # Use the (2, 3) grid slot for the legend
    # # legend_ax.axis("off")  # Hide axes for the legend box
    # # legend_ax.legend(*ax.get_legend_handles_labels(), loc='center', frameon=False)
    
    # #ACCESSORIES
    # plt.suptitle('(ALL, SHALLOW < 4 km, DEEP > 6 km) Tracked Lagrangian SBZ (BL-LFC + 20 minutes) Parcels')
    # plt.tight_layout()
    
    # #MATCHING X AXISES WITH NEXT PLOT
    # #(TO KEEP AXISES CONSISTENT RUN PLOTS IN ORDER: 1st Plots, 2nd Plots, 1st Plots)
    # for axis in axs:
    #     nbins=7;axis.xaxis.set_major_locator(MaxNLocator(nbins=nbins))
    # if 'ax8' in globals():
    #     fix_x_limits([ax1,ax8])
    #     fix_x_limits([ax2,ax9])
    #     fix_x_limits([ax3,ax10])
    #     fix_x_limits([ax4,ax11])
    #     fix_x_limits([ax5,ax12])
    #     fix_x_limits([ax6,ax13])
    #     fix_x_limits([ax7,ax14])
    #     del ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10,ax11,ax12,ax13,ax14


In [ ]:
if plotting==True:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    
    
    # List of profile arrays and their corresponding labels and x-axis titles
    profiles = [
        (ALL_ColdPool_profile_array_VMF_c.copy(), SHALLOW_ColdPool_profile_array_VMF_c.copy(), DEEP_ColdPool_profile_array_VMF_c.copy(), 'VMF_c (kg/m^2/s)'),
        (ALL_ColdPool_profile_array_VMF_g.copy(), SHALLOW_ColdPool_profile_array_VMF_g.copy(), DEEP_ColdPool_profile_array_VMF_g.copy(), 'VMF_g (kg/m^2/s)')
    ]
    
    
    
    # Set up the 2x3 gridspec
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    
    # Loop through the profiles and plot them on subplots
    # for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
    for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
    
        ax = fig.add_subplot(gs[i])
     
        # Plot Profiles
        profile_list=[ALL_profile_array,SHALLOW_profile_array,DEEP_profile_array]
        colors=['black','green','blue']
        labels=['ALL','SHALLOW','DEEP']
        for (profile,color,label) in zip(profile_list,colors,labels):   
            profile=averaged_profiles(profile) # Compute the averaged profile
            
            ax.plot(profile[:, 0], profile[:, 1],color=color,label=label)
            
        ax.set_xlabel(xlabel)
        ax.set_ylabel('z (km)')
        ax.grid(True)
    
        #LEGEND
        if i==1:
            ax.legend(loc='upper center')
    
        apply_scientific_notation([ax])
    
    #MEAN CLOUD BASE
    axs = fig.get_axes()
    ax1=axs[0]
    for axis in [ax1]:
        axis.axhline(all_cloudbase,color='purple',linestyle='dashed')
     
    # #LEGEND
    # # legend_ax = fig.add_subplot(gs[1, 2])  # Use the (2, 3) grid slot for the legend
    # # legend_ax.axis("off")  # Hide axes for the legend box
    # # legend_ax.legend(*ax.get_legend_handles_labels(), loc='center', frameon=False)
    
    # #ACCESSORIES
    # plt.suptitle('(ALL, SHALLOW < 4 km, DEEP > 6 km) Tracked Lagrangian ColdPool (BL-LFC + 20 minutes) Parcels')
    # plt.tight_layout()
    
    # #MATCHING X AXISES WITH NEXT PLOT
    # #(TO KEEP AXISES CONSISTENT RUN PLOTS IN ORDER: 1st Plots, 2nd Plots, 1st Plots)
    # for axis in axs:
    #     nbins=7;axis.xaxis.set_major_locator(MaxNLocator(nbins=nbins))
    # if 'ax8' in globals():
    #     fix_x_limits([ax1,ax8])
    #     fix_x_limits([ax2,ax9])
    #     fix_x_limits([ax3,ax10])
    #     fix_x_limits([ax4,ax11])
    #     fix_x_limits([ax5,ax12])
    #     fix_x_limits([ax6,ax13])
    #     fix_x_limits([ax7,ax14])
    #     del ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10,ax11,ax12,ax13,ax14


In [ ]:
if plotting==True:
    #SBZ vs ColdPool
    def average_difference(array1, array2):
        out_var_one=averaged_profiles(array1)
        out_var_two=averaged_profiles(array2)
    
        #masking out non matches
        second_col_one = out_var_one[:, 1]
        second_col_two = out_var_two[:, 1]
        mask_one = np.isin(second_col_one, second_col_two)
        mask_two = np.isin(second_col_two, second_col_one)
        
        out_var_one = out_var_one.copy()[mask_one]
        out_var_two = out_var_two.copy()[mask_two]
        
        diff=(out_var_one[:,0]-out_var_two[:,0])
        zs=out_var_one[:,1]
    
        out_profile=np.zeros((len(diff),2))
    
        out_profile[:,0]=diff;out_profile[:,1]=zs;
        return out_profile
        
    # Calculate average differences for each variable (w, qv, qc, th, buoyancy) for ColdPool
    
    print('VMF_c')
    ALL_diff_VMF_c = average_difference(ALL_profile_array_VMF_c, ALL_ColdPool_profile_array_VMF_c)
    SHALLOW_diff_VMF_c = average_difference(SHALLOW_profile_array_VMF_c, SHALLOW_ColdPool_profile_array_VMF_c)
    DEEP_diff_VMF_c = average_difference(DEEP_profile_array_VMF_c, DEEP_ColdPool_profile_array_VMF_c)
    
    print('VMF_g')
    ALL_diff_VMF_g = average_difference(ALL_profile_array_VMF_g, ALL_ColdPool_profile_array_VMF_g)
    SHALLOW_diff_VMF_g = average_difference(SHALLOW_profile_array_VMF_g, SHALLOW_ColdPool_profile_array_VMF_g)
    DEEP_diff_VMF_g = average_difference(DEEP_profile_array_VMF_g, DEEP_ColdPool_profile_array_VMF_g)

In [ ]:
if plotting==True:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    
    # List of profile arrays and their corresponding labels and x-axis titles
    profiles = [
        (ALL_diff_VMF_c.copy(), SHALLOW_diff_VMF_c.copy(), DEEP_diff_VMF_c.copy(), 'VMF_c (kg/m^2/s)'),
        (ALL_diff_VMF_g.copy(), SHALLOW_diff_VMF_g.copy(), DEEP_diff_VMF_g.copy(), 'VMF_g (kg/m^2/s)')
    ]
    
    
    # Set up the 2x3 gridspec
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    
    # Loop through the profiles and plot them on subplots
    for i, (ALL_profile_array, SHALLOW_profile_array, DEEP_profile_array, xlabel) in enumerate(profiles):
        ax = fig.add_subplot(gs[i])
    
        
        # Compute the averaged profile
        ALL_profile = ALL_profile_array.copy()
        SHALLOW_profile = SHALLOW_profile_array.copy()
        DEEP_profile = DEEP_profile_array.copy()
    
        if xlabel in [r'$q_v$ (g/kg)',r'$q_c+q_i$ (g/kg)','HMC (g/kg/s)']:
            ALL_profile[:,0]*=1000
            SHALLOW_profile[:,0]*=1000
            DEEP_profile[:,0]*=1000
        
        # Plot the profile
        ax.plot(ALL_profile[:, 0], ALL_profile[:, 1],color='black',label='ALL SBZ-ColdPool')
        ax.plot(SHALLOW_profile[:, 0], SHALLOW_profile[:, 1],color='green',label='SHALLOW SBZ-ColdPool')
        ax.plot(DEEP_profile[:, 0], DEEP_profile[:, 1],color='blue',label='DEEP SBZ-ColdPool')
        ax.set_xlabel(xlabel)
        ax.set_ylabel('z (km)')
        ax.grid(True)
    
        ax.axvline(0,color='k',linestyle='dashed')
    
        #LEGEND
        if i==1:
            ax.legend(loc='upper left')
    
        apply_scientific_notation([ax])
    
    
    #MEAN CLOUD BASE
    axs = fig.get_axes()
    ax1 = axs[0]
    for axis in [ax1]:
        axis.axhline(all_cloudbase,color='purple',linestyle='dashed')
        
    # #LEGEND
    # legend_ax = fig.add_subplot(gs[1, 2])  # Use the (2, 3) grid slot for the legend
    # legend_ax.axis("off")  # Hide axes for the legend box
    # legend_ax.legend(*ax.get_legend_handles_labels(), loc='center', frameon=False)
    
    #ACCESSORIES
    plt.suptitle('(ALL, SHALLOW < 4 km, DEEP > 6 km) Tracked Lagrangian SBZ-ColdPool Difference (BL-LFC + 30 minutes) Parcels')
    plt.tight_layout()
    
